Prueba

In [1]:
pip install hyperopt

Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from hyperopt import hp, fmin, tpe, Trials, STATUS_OK
from keras.models import Sequential
from keras.layers import Dense, Dropout, GRU, InputLayer, Flatten
from keras.callbacks import EarlyStopping, ModelCheckpoint
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.optimizers import Adam
import os
np.random.seed(42)


LEER DATASET

In [3]:
import pandas as pd

datos = pd.read_csv("C:\\Users\\wamt1\\OneDrive\\Escritorio\\colab\\SNpollutionFinal.csv")

datos['date'] = pd.to_datetime(datos['date'])

# Se establece la columna date como index
datos.set_index('date', inplace=True)


In [4]:
datos.head()

,pollution,dew,temp,press,wnd_dir,wnd_spd
date,,,,,,
2010-01-01 00:00:00,129.0,-21,-11.0,1021.0,2,1.79
2010-01-01 01:00:00,129.0,-21,-12.0,1020.0,2,4.92
2010-01-01 02:00:00,129.0,-21,-11.0,1019.0,2,6.71
2010-01-01 03:00:00,129.0,-21,-14.0,1019.0,2,9.84
2010-01-01 04:00:00,129.0,-20,-12.0,1018.0,2,12.97


Se eliminan las primeras 24 filas debido a que estas contenían valores NAN en la columna pollution, y habían sido rellenadas con interpolación lineal.

In [5]:
datos = datos.drop(datos.index[:24])

datos

,pollution,dew,temp,press,wnd_dir,wnd_spd
date,,,,,,
2010-01-02 00:00:00,129.0,-16,-4.0,1020.0,1,1.79
2010-01-02 01:00:00,148.0,-15,-4.0,1020.0,1,2.68
2010-01-02 02:00:00,159.0,-11,-5.0,1021.0,1,3.57
2010-01-02 03:00:00,181.0,-7,-5.0,1022.0,1,5.36
2010-01-02 04:00:00,138.0,-7,-5.0,1022.0,1,6.25
...,...,...,...,...,...,...
2014-12-31 19:00:00,8.0,-23,-2.0,1034.0,2,231.97
2014-12-31 20:00:00,10.0,-22,-3.0,1034.0,2,237.78
2014-12-31 21:00:00,10.0,-22,-3.0,1034.0,2,242.70


In [6]:
#Se verifica que el index está en formato de dato "datetime64"
print("El index del dataframe input es un tipo de dato: ", datos.index.dtype)


El index del dataframe input es un tipo de dato:  datetime64[ns]


Se divide el dataset

In [7]:
# Dividir el conjunto de datos en entrenamiento y prueba
train, test = train_test_split(datos, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
test, val = train_test_split(test, test_size=0.33, shuffle=False)

print("Las dimensiones de train son: ", train.shape)
print("Las dimensiones de test son: ", test.shape)
print("Las dimensiones de val son: ", val.shape)

Las dimensiones de train son:  (30660, 6)
Las dimensiones de test son:  (8803, 6)
Las dimensiones de val son:  (4337, 6)


Se normalizan los datos

In [8]:
from sklearn.preprocessing import StandardScaler


# Normalizar solo con los datos de entrenamiento
scaler = StandardScaler()
train = scaler.fit_transform(train)

# Aplicar la transformación a test y val usando los parámetros de train
test = scaler.transform(test)
val = scaler.transform(val)

train = pd.DataFrame(train, columns=datos.columns)
test = pd.DataFrame(test, columns=datos.columns)
val = pd.DataFrame(val, columns=datos.columns)


Se unen los datos nuevamente, ahora normalizados, en un único conjunto.

In [9]:
datosNormalizados = pd.concat([train, test, val])

datosNormalizados.index = datos.index


In [10]:
datosNormalizados.shape

(43800, 6)

In [11]:
datosNormalizados.head(10)


,pollution,dew,temp,press,wnd_dir,wnd_spd
date,,,,,,
2010-01-02 00:00:00,0.317681,-1.214023,-1.268524,0.329687,-0.380944,-0.464048
2010-01-02 01:00:00,0.526152,-1.144302,-1.268524,0.329687,-0.380944,-0.446575
2010-01-02 02:00:00,0.646846,-0.865419,-1.349314,0.426127,-0.380944,-0.429103
2010-01-02 03:00:00,0.888234,-0.586536,-1.349314,0.522567,-0.380944,-0.393962
2010-01-02 04:00:00,0.416431,-0.586536,-1.349314,0.522567,-0.380944,-0.376489
2010-01-02 05:00:00,0.098238,-0.586536,-1.430104,0.522567,-0.380944,-0.359017
2010-01-02 06:00:00,0.054349,-0.586536,-1.430104,0.619008,-0.380944,-0.323876
2010-01-02 07:00:00,0.262820,-0.586536,-1.349314,0.715448,-0.380944,-0.288735
2010-01-02 08:00:00,0.218931,-0.656257,-1.430104,0.715448,-0.380944,-0.253594


Espacio de búsqueda

In [12]:
space = {
    'layers': hp.quniform('layers', 1, 4, 1), # Cantidad de capas GRU
    'units': hp.choice('units', [2 ** i for i in range(3, 8)]),  # Número de unidades GRU
    'activation': hp.choice('activation', ['tanh', 'sigmoid', 'relu', 'linear']),
    'dropout': hp.quniform('dropout', 0, 0.5, 0.1),  # Dropout para regularización
    'learning_rate': hp.loguniform('learning_rate', np.log(0.000001), np.log(0.01)),  # Tasa de aprendizaje
    'epochs': hp.choice('epochs', [2 ** i for i in range(3, 9)]),  # Número de épocas de entrenamiento
    'batch': hp.choice('batch',[2 ** i for i in range(3, 9)])
}

Se establece el formato de datos de entrada para redes GRU, es decir [observaciones, retardos, caracteristicas]

In [13]:
futuros = 1
pasados  = 12

In [14]:
datosX = []
datosY = []
for i in range(pasados, len(datosNormalizados) - futuros + 1):
  datosX.append(datosNormalizados.iloc[i-pasados:i, 0:datosNormalizados.shape[1]])
  datosY.append(datosNormalizados.iloc[i+futuros-1:i+futuros, 0])


In [15]:
# Convertir las listas en arrays numpy
datosX = np.array(datosX)
datosY = np.array(datosY)

# Ver las dimensiones (shape) de los arrays
print("Dimensiones de X:", datosX.shape)  # (n_muestras, pasos_de_tiempo, n_características)
print("Dimensiones de Y:", datosY.shape)  # (n_muestras, n_características)

Dimensiones de X: (43788, 12, 6)
Dimensiones de Y: (43788, 1)


In [16]:
inputs = datosX.shape[1] * datosX.shape[2]
datosX = datosX.reshape(datosX.shape[0], inputs)

In [17]:
print("Dimensiones de X después de rehape:", datosX.shape)

Dimensiones de X después de rehape: (43788, 72)


Se dividen nuevamente los conjuntos de datos

In [18]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto de datos en entrenamiento y prueba
trainX, testX = train_test_split(datosX, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
testX, valX = train_test_split(testX, test_size=0.33, shuffle=False)

print("Las dimensiones de trainX son: ", trainX.shape)
print("Las dimensiones de testX son: ", testX.shape)
print("Las dimensiones de valX son: ", valX.shape)


Las dimensiones de trainX son:  (30651, 72)
Las dimensiones de testX son:  (8801, 72)
Las dimensiones de valX son:  (4336, 72)


In [19]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto de datos en entrenamiento y prueba
trainY, testY = train_test_split(datosY, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
testY, valY = train_test_split(testY, test_size=0.33, shuffle=False)

print("Las dimensiones de trainY son: ", trainY.shape)
print("Las dimensiones de testY son: ", testY.shape)
print("Las dimensiones de valY son: ", valY.shape)

Las dimensiones de trainY son:  (30651, 1)
Las dimensiones de testY son:  (8801, 1)
Las dimensiones de valY son:  (4336, 1)


Se crean métricas para medir desempeño

In [20]:
import tensorflow.keras.backend as K

def smape(y_true, y_pred):
    """
    Define la función SMAPE (Error Porcentual Absoluto Medio Simétrico).
    """
    summ = K.abs(y_true) + K.abs(y_pred)
    smape_val = K.abs(y_pred - y_true) / summ * 2.0
    return K.mean(smape_val, axis=-1)

def rmse(y_true, y_pred):
    return K.sqrt(K.mean(K.square(y_pred - y_true)))

def ia(y_true, y_pred):
    numerator = K.sum(K.abs(y_true - y_pred))
    denominator = K.sum(K.abs(y_true - K.mean(y_true)) + K.abs(y_pred - K.mean(y_true)))
    return 1 - (numerator / denominator)

In [21]:
from sklearn.metrics import mean_squared_error as mse
from sklearn.metrics import mean_absolute_error as mae

def graficarPrediccion(modelo, x, y, inicio, final):
  predicciones = modelo.predict(x)
  predicciones = predicciones.flatten()
  df = pd.DataFrame({'Originales': y, 'Predichos': predicciones})
  plt.plot(df.index, df['Originales'][inicio:final], label='Originales')
  plt.plot(df.index, df['Predichos'][inicio:final], label='Predichos')
  return df, mse(y, predicciones),  mae(y, predicciones), rmse(y, predicciones), smape(y,predicciones), ia(y, predicciones)

Versión Final


In [22]:
def objective(params):

    model = Sequential()
    model.add(InputLayer(input_shape=(trainX.shape[1],)))
    if (params['layers'] == 1):
      model.add(Dense(units=params['units'], activation=params['activation']))
      model.add(Dropout(params['dropout']))

    else:
      for _ in range(int(params['layers']) - 1):
          model.add(Dense(units=params['units'], activation=params['activation']))
          model.add(Dropout(params['dropout']))
      model.add(Dense(units=params['units'], activation=params['activation']))
      model.add(Dropout(params['dropout']))

    model.add(Dense(1))


    opt = Adam(learning_rate=params['learning_rate'])
    model.compile(optimizer=opt, loss='mse', metrics=["mae", smape, rmse, ia])

    early_stopping = EarlyStopping(monitor='val_loss', patience=15, verbose=1, restore_best_weights= True)

    model.fit(trainX, trainY, epochs=params['epochs'],
                        validation_data=(testX, testY),
                        verbose = 2, batch_size=params['batch'], callbacks=[early_stopping])


    predictions = model.predict(valX)


    loss = mean_squared_error(valY, predictions)

    return {'loss': loss, 'status': STATUS_OK}

In [23]:
trials = Trials()
best = fmin(objective, space, algo=tpe.suggest, max_evals=50, trials=trials, rstate=np.random.default_rng(42))

print(best)

  0%|          | 0/50 [00:00<?, ?trial/s, best loss=?]

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                           

958/958 - 9s - 9ms/step - ia: 0.3315 - loss: 1.5169 - mae: 0.8856 - rmse: 1.1891 - smape: 1.3978 - val_ia: 0.2779 - val_loss: 0.6564 - val_mae: 0.6074 - val_rmse: 0.6891 - val_smape: 1.3486

Epoch 2/128                                           

958/958 - 2s - 3ms/step - ia: 0.4725 - loss: 0.7613 - mae: 0.6317 - rmse: 0.8491 - smape: 1.2168 - val_ia: 0.4203 - val_loss: 0.2952 - val_mae: 0.4005 - val_rmse: 0.4707 - val_smape: 0.8611

Epoch 3/128                                           

958/958 - 2s - 3ms/step - ia: 0.5821 - loss: 0.5244 - mae: 0.5139 - rmse: 0.7032 - smape: 1.0360 - val_ia: 0.5003 - val_loss: 0.1954 - val_mae: 0.3219 - val_rmse: 0.3862 - val_smape: 0.7006

Epoch 4/128                                           

958/958 - 3s - 3ms/step - ia: 0.6293 - loss: 0.4354 - mae: 0.4613 - rmse: 0.6395 - smape: 0.9411 - val_ia: 0.5244 - val_loss: 0.1769 - val_mae: 0.3043 - val_rmse: 0.3654 - val_smape: 0.6538

Epoch 5/128 

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



120/120 - 9s - 71ms/step - ia: 0.7700 - loss: 0.2170 - mae: 0.3027 - rmse: 0.4412 - smape: 0.6677 - val_ia: 0.8184 - val_loss: 0.0931 - val_mae: 0.2018 - val_rmse: 0.2922 - val_smape: 0.4989

Epoch 2/16                                                                          

120/120 - 1s - 8ms/step - ia: 0.8644 - loss: 0.0936 - mae: 0.1975 - rmse: 0.3014 - smape: 0.4750 - val_ia: 0.8419 - val_loss: 0.0761 - val_mae: 0.1785 - val_rmse: 0.2619 - val_smape: 0.4477

Epoch 3/16                                                                          

120/120 - 1s - 12ms/step - ia: 0.8753 - loss: 0.0808 - mae: 0.1821 - rmse: 0.2814 - smape: 0.4477 - val_ia: 0.8463 - val_loss: 0.0757 - val_mae: 0.1778 - val_rmse: 0.2596 - val_smape: 0.4387

Epoch 4/16                                                                          

120/120 - 1s - 9ms/step - ia: 0.8817 - loss: 0.0747 - mae: 0.1731 - rmse: 0.2694 - smape: 0.4315 - val_ia: 0.8471 - val_loss: 0.0721 - val_mae: 0.1765 - val_rmse: 0.25

...

Epoch 1/16                                                                           

240/240 - 10s - 41ms/step - ia: 0.7615 - loss: 0.2936 - mae: 0.3338 - rmse: 0.4859 - smape: 0.6780 - val_ia: 0.7987 - val_loss: 0.1018 - val_mae: 0.2042 - val_rmse: 0.2956 - val_smape: 0.4940

Epoch 2/16                                                                           

240/240 - 2s - 7ms/step - ia: 0.8661 - loss: 0.0996 - mae: 0.1936 - rmse: 0.3078 - smape: 0.4605 - val_ia: 0.8337 - val_loss: 0.0770 - val_mae: 0.1703 - val_rmse: 0.2558 - val_smape: 0.4296

Epoch 3/16                                                                           

240/240 - 2s - 6ms/step - ia: 0.8809 - loss: 0.0831 - mae: 0.1731 - rmse: 0.2804 - smape: 0.4216 - val_ia: 0.8435 - val_loss: 0.0683 - val_mae: 0.1589 - val_rmse: 0.2405 - val_smape: 0.4055

Epoch 4/16                                                                           

240/240 - 1s - 6ms/step - ia: 0.8893 - loss: 0.0747 - mae: 0.1612 - rmse: 0.2

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                                                          

120/120 - 9s - 73ms/step - ia: 0.3767 - loss: 2.1730 - mae: 1.0718 - rmse: 1.4714 - smape: 1.2229 - val_ia: 0.3982 - val_loss: 1.7822 - val_mae: 0.9018 - val_rmse: 1.1779 - val_smape: 1.0672

Epoch 2/128                                                                          

120/120 - 1s - 7ms/step - ia: 0.3780 - loss: 2.1541 - mae: 1.0655 - rmse: 1.4640 - smape: 1.2217 - val_ia: 0.4012 - val_loss: 1.7438 - val_mae: 0.8924 - val_rmse: 1.1641 - val_smape: 1.0662

Epoch 3/128                                                                          

120/120 - 2s - 13ms/step - ia: 0.3816 - loss: 2.0859 - mae: 1.0521 - rmse: 1.4419 - smape: 1.2200 - val_ia: 0.4042 - val_loss: 1.7069 - val_mae: 0.8832 - val_rmse: 1.1510 - val_smape: 1.0651

Epoch 4/128                                                                          

120/120 - 2s - 15ms/step - ia: 0.3842 - loss: 2.0570 - mae: 1.0429 - rmse: 1.

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/32                                                                           

958/958 - 17s - 18ms/step - ia: 0.4675 - loss: 0.7290 - mae: 0.6149 - rmse: 0.8228 - smape: 1.1666 - val_ia: 0.4877 - val_loss: 0.2828 - val_mae: 0.3495 - val_rmse: 0.4384 - val_smape: 0.7387

Epoch 2/32                                                                           

958/958 - 4s - 5ms/step - ia: 0.6968 - loss: 0.3537 - mae: 0.4237 - rmse: 0.5784 - smape: 0.8130 - val_ia: 0.5968 - val_loss: 0.1745 - val_mae: 0.2565 - val_rmse: 0.3379 - val_smape: 0.5700

Epoch 3/32                                                                           

958/958 - 4s - 4ms/step - ia: 0.7464 - loss: 0.2557 - mae: 0.3546 - rmse: 0.4887 - smape: 0.7211 - val_ia: 0.6251 - val_loss: 0.1305 - val_mae: 0.2251 - val_rmse: 0.2999 - val_smape: 0.5243

Epoch 4/32                                                                           

958/958 - 5s - 5ms/step - ia: 0.7736 - loss: 0.2023 - mae: 0.3132 - rmse: 0.4

In [24]:
print(best)

{'activation': 1, 'batch': 2, 'dropout': 0.1, 'epochs': 4, 'layers': 1.0, 'learning_rate': 0.0001524958564276629, 'units': 4}
